#Environment Variables


In [1]:
import os

os.environ["HF_TOKEN"]="Your HF-TOKEN"
os.environ["COMET_API_KEY"]="Your COMET-API-KEY"

#Imports

In [3]:
import comet_ml
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import DPOTrainer,DPOConfig
from transformers import TextStreamer
from datasets import load_dataset

#Login to Comet ML

In [4]:
comet_ml.login()

In [5]:
exp=comet_ml.start(project_name="llm_twin_fine_tuning_dpo_model")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/spiralmonster/llm-twin-fine-tuning-dpo-model/6ce4e1b55bfb40b3b910c625b4623726



#Load Model

In [7]:
HF_USERNAME="spiralMon"
model_name="Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit"
model_id=HF_USERNAME+"/"+model_name

In [8]:
max_seq_length=2048

In [9]:
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    device_map="cuda"
)

==((====))==  Unsloth 2026.8.14: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load spiralMon/Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.


# Create PEFT Model

In [10]:
lora_rank=32
lora_alpha=32
lora_dropout=0
target_modules=[
    "q_proj",
    "k_proj",
    "v_proj",
    "up_proj",
    "down_proj",
    "o_proj",
    "gate_proj"
]

In [12]:
model=FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    target_modules=target_modules
)

Unsloth 2026.8.14 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


#Load Dataset

In [13]:
dataset_name="llm_twin_preference_dataset"
dataset_id=HF_USERNAME+"/"+dataset_name

In [14]:
dataset=load_dataset(dataset_id)

README.md:   0%|          | 0.00/369 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  823kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2163 [00:00<?, ? examples/s]

In [15]:
dataset=dataset["train"]

In [16]:
dataset

Dataset({
    features: ['instructions', 'chosen_answers', 'rejected_answers'],
    num_rows: 2163
})

#Format Dataset

In [17]:
alpaca_template="""
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
"""

In [18]:
EOS_TOKEN=tokenizer.eos_token

In [19]:
def format_samples(example):
  prompt=alpaca_template.format(
      example["instructions"]
  )
  chosen=example["chosen_answers"]+EOS_TOKEN
  rejected=example["rejected_answers"]+EOS_TOKEN

  result={
      "prompt":prompt,
      "chosen":chosen,
      "rejected":rejected
  }

  return result

In [20]:
dataset=dataset.map(format_samples)

Map:   0%|          | 0/2163 [00:00<?, ? examples/s]

In [22]:
dataset=dataset.remove_columns([
    "instructions",
    "chosen_answers",
    "rejected_answers"
])

In [24]:
dataset=dataset.train_test_split(
    test_size=0.05
)

In [25]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 2054
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 109
    })
})

#Model Fine-Tuning

In [27]:
training_arguments=DPOConfig(
    learning_rate=2e-6,
    lr_scheduler_type="linear",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    warmup_steps=10,
    output_dir="output",
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=1,
    report_to="comet_ml",
    seed=0
)

In [28]:
trainer=DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.5,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    max_length=max_seq_length//2,
    max_prompt_length=max_seq_length//2,
    args=training_arguments
)

Extracting prompt in train dataset (num_proc=2):   0%|          | 0/2054 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=2):   0%|          | 0/2054 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/2054 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=2):   0%|          | 0/109 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=2):   0%|          | 0/109 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=2):   0%|          | 0/109 [00:00<?, ? examples/s]

In [29]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,054 | Num Epochs = 3 | Total steps = 387
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)
COMET INFO: An experiment with the same configuration options is already running and will be reused.
COMET WARNING: String value length exceeds 1000 characters and will be truncated. Provided value: 'LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'LlamaForCausalLM', 'parent_library': 'transformers.models.llama.modeling_llama', 'unsloth_fixed': True}, peft_version='0.19.1', base_model_name_or_path='spiralMon/Twin-LLM-Fine-Tuned-Instruct-Model-Llama-3.1-8B-bnb-4bit', revision=None, inference_mode=False, r=32, target_modules={'q_proj', 'k_proj', 'do

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
78,0.136090,0.146247,2.646624,0.230124,0.990909,2.416500,-224.209137,-110.315224,-1.689843,-1.591705
156,0.041676,0.041886,6.078953,0.449522,1.000000,5.629431,-217.344498,-109.876427,-1.703658,-1.621370


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
78,0.136090,0.146247,2.646624,0.230124,0.990909,2.416500,-224.209137,-110.315224,-1.689843,-1.591705
156,0.041676,0.041886,6.078953,0.449522,1.000000,5.629431,-217.344498,-109.876427,-1.703658,-1.621370
234,0.014046,0.025799,6.265646,-0.806937,1.000000,7.072584,-216.971115,-112.389343,-1.701510,-1.617281
312,0.011108,0.020241,6.916384,-0.885485,1.000000,7.801869,-215.669632,-112.546448,-1.700381,-1.617886
387,0.005392,0.019348,7.187113,-0.893229,1.000000,8.080341,-215.128159,-112.561913,-1.699406,-1.616368


Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-387/tokenizer_config.json.


TrainOutput(global_step=387, training_loss=0.10512414960869292, metrics={'train_runtime': 8499.0591, 'train_samples_per_second': 0.725, 'train_steps_per_second': 0.046, 'total_flos': 0.0, 'train_loss': 0.10512414960869292, 'epoch': 3.0})

In [30]:
exp.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : northern_granite_5475
COMET INFO:     url                   : https://www.comet.com/spiralmonster/llm-twin-fine-tuning-dpo-model/6ce4e1b55bfb40b3b910c625b4623726
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     eval/logits/chosen [5]         : (-1.7036583423614502, -1.6898434162139893)
COMET INFO:     eval/logits/rejected [5]       : (-1.6213696002960205, -1.5917046070098877)
COMET INFO:     eval/logps/chosen [5]          : (-224.20913696289062, -215.12815856933594)
COMET INFO:     eval/logps/rejected [5]        : (-112.5619125366211, -109.87642669677734)
COMET INFO:     eval/loss [5]                  : (0.019348029047250748, 0.1462465822696

#Testing the Fine-Tuned Model

In [31]:
test_model=FastLanguageModel.for_inference(model)

In [36]:
input_prompt=alpaca_template.format(
    "Write an article about how to find a good partner?",
    ""
)

In [37]:
model_input=tokenizer([input_prompt],return_tensors="pt").to("cuda")

In [38]:
text_streamer=TextStreamer(tokenizer)

In [39]:
output=model.generate(
    **model_input,
    streamer=text_streamer,
    max_new_tokens=256,
    use_cache=True
)

<|begin_of_text|>
Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
Write an article about how to find a good partner?

### Response:


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The search for a good partner is a complex and often frustrating process. However, the key to finding a good partner is not in the search itself, but in the search for the search. The search for the search is a process of self-reflection and self-discovery, where one must first understand their own values, needs, and priorities before they can effectively evaluate potential partners. This requires a deep understanding of one's own desires and what truly matters to them, and a willingness to be honest and transparent about these things. Only then can one truly find a partner who aligns with their values and goals, and who can provide the support and stability they need in life.
<|end_of_text|>


#Publishing Model to Hugging Face Hub

In [40]:
model_name="Twin-LLM-Fine-Tuned-DPO-Model-Llama-3.1-8B"
model_id=HF_USERNAME+"/"+model_name

In [41]:
model.push_to_hub_merged(
    model_id,
    tokenizer,
    save_method="merged_16bit"
)

Unsloth: Restored added_tokens_decoder metadata in spiralMon/Twin-LLM-Fine-Tuned-DPO-Model-Llama-3.1-8B/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...



Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [09:54<03:20, 200.86s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [10:20<00:00, 155.10s/it]


Unsloth: Merge process complete. Saved to `/content/spiralMon/Twin-LLM-Fine-Tuned-DPO-Model-Llama-3.1-8B`
